# 04. Inference, Intervals, and Prediction

Regression output gives a fitted line, but decisions usually need uncertainty. This chapter covers slope tests, coefficient confidence intervals, mean-response confidence intervals, and prediction intervals.

This chapter separates two interval types:

- confidence interval for the mean response at a given predictor value,
- prediction interval for a new individual observation at that predictor value.

Prediction intervals are wider because individual observations vary around the mean.

## Fit the model again

## Browser package setup

Run the next cell before the first analysis cell. It loads the scientific Python packages used in this module into the browser-based Python kernel.

The first run in a browser session can take a little time. Later notebooks usually load faster because the browser can reuse cached packages.


In [ ]:
from lite_setup import ensure_packages
await ensure_packages()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

from checks import check, check_close, check_interval_order

fuel = pd.read_csv("data/fuelcons.csv")
model = ols("Fuelcons ~ Temp", data=fuel).fit()

## Testing whether the slope is useful

The first regression hypothesis test usually asks whether the slope is zero:

$$
H_0: \beta_1 = 0
$$

against

$$
H_a: \beta_1 \neq 0.
$$

If $\beta_1=0$, the mean response does not change linearly with the predictor. If we reject $H_0$, we have evidence that the predictor is linearly related to the response.

In [ ]:
slope_estimate = model.params["Temp"]
slope_se = model.bse["Temp"]
slope_t = model.tvalues["Temp"]
slope_p = model.pvalues["Temp"]
slope_ci = model.conf_int().loc["Temp"]

print(f"slope estimate = {slope_estimate:.4f}")
print(f"slope standard error = {slope_se:.4f}")
print(f"t statistic = {slope_t:.3f}")
print(f"p-value = {slope_p:.6f}")
print(f"95% CI for slope = ({slope_ci[0]:.4f}, {slope_ci[1]:.4f})")

In [ ]:
alpha = 0.05
print(check(
    slope_p < alpha,
    "The slope p-value is below 0.05, so the fitted linear relationship is statistically significant at the 5% level.",
    "The p-value is not below 0.05. Recheck that you are reading the Temp row, not the Intercept row."
))
print(check_close("slope p-value", slope_p, expected=0.00033, tolerance=0.00005))

Interpretation: we estimate that each one-unit increase in temperature is associated with a decrease in mean weekly fuel consumption between about 0.085 and 0.171 units, using a 95% confidence interval for the slope.

## ANOVA table and F-test

For simple linear regression, the F-test for the regression model answers the same significance question as the two-sided slope t-test.

The total variation in the response is split into:

$$
SST = SSR + SSE,
$$

where:

- $SST$ is total variation around $\bar y$,
- $SSR$ is variation explained by the regression,
- $SSE$ is unexplained residual variation.

In [ ]:
anova_table = anova_lm(model)
anova_table

In [ ]:
f_stat = anova_table.loc["Temp", "F"]
f_p = anova_table.loc["Temp", "PR(>F)"]

print(f"F statistic = {f_stat:.3f}")
print(f"F-test p-value = {f_p:.6f}")
print(check_close("F statistic", f_stat, expected=53.695, tolerance=0.02))

## Get intervals for the observed data

`summary_frame()` returns the fitted mean, the confidence interval for the mean, and the prediction interval for individual observations.

In [ ]:
pred = model.get_prediction(fuel[["Temp"]])
intervals = pred.summary_frame(alpha=0.05)
intervals

Important columns:

- `mean`: fitted mean response
- `mean_ci_lower`, `mean_ci_upper`: confidence interval for the mean response
- `obs_ci_lower`, `obs_ci_upper`: prediction interval for a new observation

## Predict at one new temperature

In [ ]:
new_temp = 40
new_data = pd.DataFrame({"Temp": [new_temp]})
prediction_40 = model.get_prediction(new_data).summary_frame(alpha=0.05)
prediction_40

In [ ]:
mean_40 = prediction_40.loc[0, "mean"]
mean_ci_lower = prediction_40.loc[0, "mean_ci_lower"]
mean_ci_upper = prediction_40.loc[0, "mean_ci_upper"]
obs_ci_lower = prediction_40.loc[0, "obs_ci_lower"]
obs_ci_upper = prediction_40.loc[0, "obs_ci_upper"]

print(check_close("predicted mean at Temp = 40", mean_40, expected=10.721, tolerance=0.02))
print(check_interval_order(mean_ci_lower, mean_40, mean_ci_upper, "confidence interval"))
print(check_interval_order(obs_ci_lower, mean_40, obs_ci_upper, "prediction interval"))

## Checkpoint: which interval is wider?

In [ ]:
mean_ci_width = mean_ci_upper - mean_ci_lower
prediction_interval_width = obs_ci_upper - obs_ci_lower

print(f"Mean confidence interval width: {mean_ci_width:.3f}")
print(f"Prediction interval width: {prediction_interval_width:.3f}")

print(check(
    prediction_interval_width > mean_ci_width,
    "Correct. The prediction interval is wider because it includes individual-observation noise.",
    "Recheck the columns. The prediction interval should use obs_ci_lower and obs_ci_upper."
))

## Compare several temperatures

Change the values in `temps_to_predict` and rerun the cell.

In [ ]:
temps_to_predict = [30, 40, 50, 60]
new_points = pd.DataFrame({"Temp": temps_to_predict})

comparison = model.get_prediction(new_points).summary_frame(alpha=0.05)
comparison.insert(0, "Temp", temps_to_predict)
comparison

## Visualize uncertainty

In [ ]:
plot_grid = pd.DataFrame({"Temp": sorted(fuel["Temp"].tolist())})
plot_intervals = model.get_prediction(plot_grid).summary_frame(alpha=0.05)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(fuel["Temp"], fuel["Fuelcons"], label="Observed")
ax.plot(plot_grid["Temp"], plot_intervals["mean"], color="red", label="Fitted mean")
ax.fill_between(
    plot_grid["Temp"],
    plot_intervals["mean_ci_lower"],
    plot_intervals["mean_ci_upper"],
    color="red",
    alpha=0.2,
    label="95% mean CI",
)
ax.fill_between(
    plot_grid["Temp"],
    plot_intervals["obs_ci_lower"],
    plot_intervals["obs_ci_upper"],
    color="gray",
    alpha=0.2,
    label="95% prediction interval",
)
ax.set_xlabel("Temperature")
ax.set_ylabel("Fuel consumption")
ax.legend()
plt.show()

## Confidence level

The `alpha` argument is the probability left outside the interval.

- `alpha=0.05` gives a 95% interval.
- `alpha=0.10` gives a 90% interval.
- `alpha=0.01` gives a 99% interval.

In [ ]:
prediction_40_90 = model.get_prediction(new_data).summary_frame(alpha=0.10)
prediction_40_99 = model.get_prediction(new_data).summary_frame(alpha=0.01)

width_90 = prediction_40_90.loc[0, "obs_ci_upper"] - prediction_40_90.loc[0, "obs_ci_lower"]
width_99 = prediction_40_99.loc[0, "obs_ci_upper"] - prediction_40_99.loc[0, "obs_ci_lower"]

print(f"90% prediction interval width: {width_90:.3f}")
print(f"99% prediction interval width: {width_99:.3f}")
print(check(width_99 > width_90, "Correct. Higher confidence requires a wider interval.", "Recheck alpha: 99% uses alpha=0.01."))

## What to remember

Use a coefficient confidence interval when the question is about an unknown parameter such as the slope. Use a mean-response confidence interval when the question is about the average response at a chosen predictor value. Use a prediction interval when the question is about a new individual outcome. For practical forecasting, prediction intervals are often the more honest answer.